In [1]:
# Cell 1: Install Unsloth (simple & clean)
!pip install unsloth
!pip install datasets sentence-transformers faiss-cpu pyngrok fastapi uvicorn

print("Done! Now go to Runtime → Restart session, then run Cell 2")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 2.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 64.8 MB/s eta 0:00:00
^C
Done! Now go to Runtime → Restart session, then run Cell 2


In [1]:
# Cell 2: Load Qwen2.5-7B-Instruct with Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model loaded with Unsloth!")
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.7.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ Model loaded with Unsloth!
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [2]:
# Cell 3: Load & unify all Egyptian legal datasets
from datasets import load_dataset, Dataset
import random
import json

SYSTEM_PROMPT = """أنت "فقيه"، مساعد قانوني ذكي متخصص في القانون المصري.
مهمتك هي الإجابة على الأسئلة القانونية بلغة عربية واضحة وبسيطة.
عند الإجابة، اذكر المواد القانونية المتعلقة بالسؤال إن وُجدت.
تنبيه: إجاباتك للأغراض التعليمية فقط وليست بديلاً عن استشارة محامٍ متخصص."""

# Load datasets
print("Loading datasets...")
ds1 = load_dataset("fr3on/eg-legal-qa", split="train")
print(f"eg-legal-qa: {len(ds1)}")

ds2 = load_dataset("Omar-youssef/QA_LAW_Egyptian_dataset", split="train")
print(f"QA_LAW_Egyptian: {len(ds2)}")

ds3 = load_dataset("tarekys5/egyptian_legal_v2", split="train")
print(f"egyptian_legal_v2: {len(ds3)}")

ds4 = load_dataset("fr3on/eg-legal-instruction-following", split="train")
print(f"eg-legal-instruction: {len(ds4)}")

ds_corpus = load_dataset("dataflare/egypt-legal-corpus", split="train")
print(f"legal corpus (RAG): {len(ds_corpus)}")

# Unify all datasets
all_data = []

for row in ds1:
    all_data.append({"instruction": row["instruction"], "input": row["input"], "output": row["output"]})

for row in ds2:
    all_data.append({"instruction": "أجب على السؤال القانوني التالي بناءً على القانون المصري", "input": row["question"], "output": row["answer"]})

for row in ds3:
    output = row["output"]
    if row.get("legal_basis") and row["legal_basis"].strip():
        output = f"السند القانوني: {row['legal_basis']}\n\n{row['output']}"
    all_data.append({"instruction": "أجب على السؤال القانوني التالي مع ذكر السند القانوني", "input": row["instruction"], "output": output})

for row in ds4:
    all_data.append({"instruction": row["instruction"], "input": row["input"], "output": row["output"]})

# Shuffle & create dataset
random.seed(42)
random.shuffle(all_data)
train_dataset = Dataset.from_list(all_data)

# Save RAG corpus
corpus_data = [{"text": r["text"], "law_name": r["law_name"], "categories": r.get("categories", [])} for r in ds_corpus]
with open("legal_corpus.json", "w", encoding="utf-8") as f:
    json.dump(corpus_data, f, ensure_ascii=False, indent=2)

print(f"\nTotal training examples: {len(train_dataset)}")
print(f"RAG corpus saved: {len(corpus_data)} documents")

📥 Loading datasets...


README.md:   0%|          | 0.00/4.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  346kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5230 [00:00<?, ? examples/s]

...

README.md:   0%|          | 0.00/3.17k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 24.9MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2434 [00:00<?, ? examples/s]

legal corpus (RAG): 2434

Total training examples: 22932
RAG corpus saved: 2434 documents


In [4]:
# Cell 4: Format data for Qwen2.5 ChatML
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        user_msg = instruction
        if input_text and input_text.strip():
            user_msg += "\n" + input_text

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": output},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text + EOS_TOKEN)
    return {"text": texts}

print("Formatting data...")
train_formatted = train_dataset.map(formatting_prompts_func, batched=True)

print(f"{len(train_formatted)} examples formatted!")
print(f"\n Sample (first 300 chars):")
print(train_formatted[0]["text"][:300] + "...")

Formatting data...


Map:   0%|          | 0/22932 [00:00<?, ? examples/s]

22932 examples formatted!

 Sample (first 300 chars):
<|im_start|>system
أنت "فقيه"، مساعد قانوني ذكي متخصص في القانون المصري. 
مهمتك هي الإجابة على الأسئلة القانونية بلغة عربية واضحة وبسيطة.
عند الإجابة، اذكر المواد القانونية المتعلقة بالسؤال إن وُجدت.
تنبيه: إجاباتك للأغراض التعليمية فقط وليست بديلاً عن استشارة محامٍ متخصص.<|im_end|>
<|im_start|>user...


In [ ]:
# Cell 5: Train with Unsloth!
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    dataset_text_field="text",
    max_seq_length=1024,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=500,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)


trainer_stats = trainer.train()

print("=" * 60)
print("TRAINING COMPLETE!")
print(f"   Steps: {trainer_stats.global_step}")
print(f"   Loss: {trainer_stats.training_loss:.4f}")
print(f"   Time: {trainer_stats.metrics['train_runtime']/60:.1f} min")

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/22932 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 22,932 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss
25,0.776735
50,0.898624
75,0.812580


In [6]:
# Cell 6: Create HuggingFace Dataset + Shuffle

import random
random.seed(42)
random.shuffle(all_data)

# Create HuggingFace Dataset
unified_dataset = Dataset.from_list(all_data)

# Split: 95% train, 5% validation
split = unified_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
val_dataset = split["test"]

print(f"Train: {len(train_dataset)} examples")
print(f"Validation: {len(val_dataset)} examples")

# Show a sample
print(f"\nSample example:")
sample = train_dataset[0]
print(f"Instruction: {sample['instruction'][:80]}...")
print(f"Input: {sample['input'][:80]}...")
print(f"Output: {sample['output'][:80]}...")

Train: 21785 examples
Validation: 1147 examples

Sample example:
Instruction: أذكر نص المادة رقم  في قانون العقوبات طبقا لأحدث التعديلات بالقانون 95 لسنة 2003...
Input: قانون العقوبات طبقا لأحدث التعديلات بالقانون 95 لسنة 2003م - المادة ...
Output: المخالفات هي الجرائم المعاقب عليها بالغرامة التي لا يزيد مقدار لها على مائة جنيه...


In [7]:
# Cell 7: Save RAG corpus for later use
corpus_data = []
for row in ds_corpus:
    corpus_data.append({
        "text": row["text"],
        "law_name": row["law_name"],
        "categories": row["categories"] if row.get("categories") else [],
    })

with open("legal_corpus.json", "w", encoding="utf-8") as f:
    json.dump(corpus_data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(corpus_data)} legal documents to legal_corpus.json")
print(f"Example law names:")
for doc in corpus_data[:5]:
    print(f"   - {doc['law_name']}")

Saved 2434 legal documents to legal_corpus.json
Example law names:
   - قوانين_الأحوال_الشخصية
   - قانون المرافعات
   - القرارات الوزارية الخاصة بالجمارك
   - موسوعة المواعيد
   - قانون الايجارات


In [8]:
# Cell 8: Check GPU
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("No GPU! Enable it in Settings")

GPU: Tesla T4 (15.6 GB)


In [9]:
# Cell 9: Load Qwen2.5-7B with Unsloth + QLoRA
from unsloth import FastLanguageModel

print("Loading Qwen2.5-7B-Instruct with 4-bit quantization...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,  # Auto detect
)

print("Setting up LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # LoRA rank
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model loaded successfully!")
print(f"   Trainable parameters: {model.print_trainable_parameters()}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


RuntimeError: operator torchvision::nms does not exist

In [ ]:
# Cell 10: Format data using Qwen2.5 ChatML template

def format_example(example):
    """Convert each example to Qwen2.5 chat format"""
    # Build user message
    user_msg = example['instruction']
    if example['input'] and example['input'].strip():
        user_msg += "\n" + example['input']

    # Create chat messages
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": example['output']},
    ]

    # Apply Qwen2.5 chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

print("Formatting training data...")
train_dataset_formatted = train_dataset.map(format_example)

print("Formatting validation data...")
val_dataset_formatted = val_dataset.map(format_example)

# Show a formatted example
print("Done!")
print(f"\n📝 Formatted example (first 500 chars):")
print(train_dataset_formatted[0]["text"][:500])
print("...")

In [ ]:
# Cell 11: Setup SFT Trainer
from trl import SFTTrainer
from transformers import TrainingArguments

print("Setting up trainer...")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset_formatted,
    eval_dataset=val_dataset_formatted,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        output_dir="outputs",
        optim="adamw_8bit",
        seed=42,
        report_to="none",
    ),
)

print("Trainer ready!")
print(f"   Batch size: 2 x 4 gradient accumulation = 8 effective")
print(f"   Train examples: {len(train_dataset_formatted)}")
print(f"   Estimated steps: ~{len(train_dataset_formatted) // 8 * 3}")
print(f"   Estimated time: ~60-90 minutes on T4")

In [ ]:
# Cell 12: START TRAINING!
print("Training started! This will take ~60-90 minutes...")
print("=" * 60)

trainer_stats = trainer.train()

print("=" * 60)
print("Training complete!")
print(f"   Total steps: {trainer_stats.global_step}")
print(f"   Training loss: {trainer_stats.training_loss:.4f}")
print(f"   Training time: {trainer_stats.metrics['train_runtime'] / 60:.1f} minutes")